### Ingest races.csv file


#### Step 1 - Read CSV file using spark dataframe reader

In [0]:
from pyspark.sql.types import StructField, StructType, IntegerType, StringType, DateType

In [0]:
races_schemas = StructType(fields=[
    StructField("raceId", IntegerType(), False),
    StructField("year", IntegerType(), True),
    StructField("round", IntegerType(), True),
    StructField("circuitId", IntegerType(), True),
    StructField("name", StringType(), True),
    StructField("date", DateType(), True),
    StructField("time", StringType(), True),
    StructField("url", StringType(), True)
])

In [0]:
races_df = spark.read \
.option("header", True) \
.schema(races_schemas) \
.csv("abfss://raw@formula1dl2026c.dfs.core.windows.net/races.csv")

#### Step 2 - Add ingestion date and race_timestamp to the dataframe

In [0]:
from pyspark.sql.functions import to_timestamp, concat, lit, current_timestamp

In [0]:
races_with_timestamp_df = races_df.withColumn("race_timestamp", to_timestamp(concat(col("date"),lit(" "), col("time")), "yyyy-MM-dd HH:mm:ss")) \
.withColumn("ingested_date", current_timestamp())

#### Step 3 - Select only the required columns & rename as required

In [0]:
from pyspark.sql.functions import col

In [0]:
races_selected_df = races_with_timestamp_df.select(col("raceId").alias("race_id"), col("year").alias("race_year"), col("round"), col("circuitId").alias("circuit_id"), col("name"), col("race_timestamp"), col("ingested_date"))

#### Step 4 - Write the output to processed container in parquet format

In [0]:
races_selected_df.write.mode("overwrite").partitionBy("race_year").parquet("abfss://processed@formula1dl2026c.dfs.core.windows.net/races")